[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke04-generativ-ai/10_bilde_tekst_clip_zero_shot_blomster.ipynb)


# 🌺 CLIP zero-shot på blomster

## Læringsmål
- Forstå bilde–tekst-felles representasjoner (CLIP)
- Utføre zero-shot klassifikasjon med tekstprompter
- Evaluere enkel nøyaktighet og inspisere topp-5 sannsynligheter


### Hvordan dette oppfyller læringsmålene

- Forstå bilde–tekst-felles representasjoner (CLIP)
  - CLIP er en multimodal modell som projiserer både bilder og tekst inn i et felles representasjonsrom. I denne notebooken ser du dette ved at vi sammenligner et bilde mot flere tekstprompter og får sannsynligheter for hvor godt teksten passer bildet. At én og samme modell kan “måle avstand” mellom bilde og tekst, illustrerer felles-representasjoner i praksis.

- Utføre zero-shot klassifikasjon med tekstprompter
  - Vi definerer klasser kun med naturlig språk (prompter som "a photo of a daisy"). Uten noen som helst fintrening på blomsterdatasettet, lar vi modellen velge den klassen hvis tekstbeskrivelse passer best til bildet. Dette er kjerneideen i zero-shot: å bruke språket som etiketter i stedet for å trene på etiketterte eksempler.

- Evaluere enkel nøyaktighet og inspisere topp-5 sannsynligheter
  - For et lite, representativt utvalg bilder beregner vi enkel accuracy (andel korrekte prediksjoner). I tillegg skriver vi ut topp-5 sannsynligheter for et eksempelbilde. Topplisten gir innsikt i modellens usikkerhet og hvilke klasser som forveksles, og er nyttig i multimodal analyse hvor “neste beste” ofte forklarer feilårsaker.

Kort sagt: du ser hvordan multimodale representasjoner muliggjør zero-shot klassifikasjon, og du lærer å evaluere resultater både med et enkelt metrikk (accuracy) og en kvalitativ topp-5-inspeksjon som belyser modellens beslutningsgrunnlag.


### Mer om CLIP  (Contrastive Language-Image Pre-training)

CLIP er en maskinlæringsmodell utviklet av OpenAI som kobler sammen bilder og tekst. Her er en kort forklaring:

**Definisjon:** CLIP (Contrastive Language-Image Pre-training) er en nevral nettverksmodell som lærer å forstå sammenhengen mellom bilder og tekstbeskrivelser ved hjelp av kontrastiv læring.

**Kontrastiv læring:** er en treningsmetode der modellen lærer ved å sammenligne lignende og ulike eksempler. Kort forklart: Modellen blir presentert for et eksempel (ankerpunkt) og lærer å gjøre liknende eksempler (positive) nærmere i vektorrommet, mens ulike eksempler (negative) blir gjort fjernere. Dette skjer ved å minimere en kontrastfunksjon som straffer modellen når den ikke skiller positive fra negative eksempler godt nok.

**Enkelt eksempel:** Hvis du viser modellen et bilde av en katt sammen med teksten "en katt", blir modellen oppmuntret til å plassere bildet og teksten nær hverandre. Samtidig blir den straffet hvis den plasserer dem nær en upassende tekstbeskrivelse som "en hund".

Fordelen er at modellen lærer meningsfull representasjon uten å trenge eksplisitt merking av hvert mulig forhold mellom eksempler.

**Hvordan det fungerer:** Modellen blir trent på millioner av bilde-tekst-par fra internett. Den lærer å representere både bilder og tekst i samme vektorrom, slik at relevante bilder og deres tekstbeskrivelser blir mappet nær hverandre, mens irrelevante kombinasjoner blir mappet langt fra hverandre.

**Praktiske bruksområder:**
- **Bildesøk:** Du kan søke etter bilder ved hjelp av naturlig språk ("en rød bil på stranden")
- **Bildeklassifisering:** Klassifisere bilder uten å være trent på spesifikke kategorier
- **Bildeteksting:** Generere tekstbeskrivelser av bilder
- **Zero-shot læring:** Modellen kan gjenkjenne objekter den aldri har sett under trening

**Fordeler:** CLIP er fleksibel og kan brukes på mange oppgaver uten å måtte retrenes spesifikt for hver enkelt oppgave. Det er også åpen kildekode og relativt tilgjengelig for eksperimentering.

I praksis er CLIP blitt en viktig byggekloss for mange moderne bildeanalyse- og genereringsmodeller.

### Teknisk/matematisk formulering av CLIP-relatert læringsmål

- Forstå bilde–tekst-felles representasjoner (CLIP)
  - CLIP består av en bildeenkoder $f_I: \mathbb{R}^{H\times W\times C} \to \mathbb{R}^d$ og en tekstenkoder $f_T: \mathcal{T} \to \mathbb{R}^d$. Begge produserer $\ell_2$-normaliserte vektorer: $\tilde{v} = v/\lVert v\rVert_2$. Likheten mellom bilde $x$ og tekstprompt $t$ er kosinuslikhet (ofte skalert):
  $$
  s(x,t) \,=\, \tau\, \langle \tilde{f}_I(x), \tilde{f}_T(t) \rangle,\quad \tau>0.
  $$
  Under pretrening optimaliseres en toveis kontrastiv (InfoNCE) loss over batch-par $(x_i,t_i)$:
  $$
  \mathcal{L} \,=\, \frac{1}{2}\bigg(\mathrm{CE}(\mathrm{softmax}(S_X), y) + \mathrm{CE}(\mathrm{softmax}(S_T), y)\bigg),
  $$
  der $S_X[i,j]=s(x_i,t_j)$, $S_T[i,j]=s(x_j,t_i)$, og $y$ er identitetslabel (riktig par). Dette driver bilder og tekster som hører sammen nærmere i det felles rommet.

- Utføre zero-shot klassifikasjon med tekstprompter
  - Gitt en etikettmengde $\mathcal{C}=\{c_1,\dots,c_M\}$ og en sett med prompt-maler $\{\pi_k\}_{k=1}^{K}$, danner vi $K$ tekstbeskrivelser per klasse: $t_{j,k}=\pi_k(c_j)$. For et bilde $x$ beregner vi sannsynligheter over alle promter med softmax:
  $$
  p(t_{j,k}\mid x) \,=\, \frac{\exp\, s(x,t_{j,k})}{\sum_{j',k'} \exp\, s(x,t_{j',k'})}.
  $$
  Klassescore aggregeres per klasse, f.eks. med gjennomsnitt:
  $$
  S_j(x) \,=\, \frac{1}{K}\sum_{k=1}^{K} p(t_{j,k}\mid x),\quad \hat{y}(x)=\arg\max_{j} S_j(x).
  $$
  (Alternativt kan man aggregere på logit-nivå før softmax eller bruke temperatur for kalibrering.)

- Evaluere enkel nøyaktighet og inspisere topp-5 sannsynligheter
  - For et testsett $\{(x_i,y_i)\}_{i=1}^{N}$ er accuracy
  $$
  \mathrm{Acc} \,=\, \frac{1}{N}\sum_{i=1}^{N} \mathbf{1}\{\hat{y}(x_i)=y_i\}.
  $$
  Topp-5 hentes ved å sortere $\{S_j(x)\}_{j=1}^M$ og vise de fem største $(c_j, S_j(x))$. Dette gir et estimat av modellens usikkerhet og typiske forvekslinger mellom klasser.

Kort sagt: CLIP realiserer et felles vektorrom hvor kosinuslikhet mellom $f_I(x)$ og $f_T(t)$ muliggjør zero-shot klassifikasjon via tekstlige etiketter; vi evaluerer med $\mathrm{Acc}$ og kvalitativ topp-5-analyse for hver $x$.


### 🔧 Miljøoppsett – fungerer både lokalt og i Google Colab


In [ ]:
import sys, subprocess, os, glob, random

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    # Lettere avhengigheter for Colab (bruk subprocess for å unngå notebook-magic avhengighet)
    try:
        import transformers  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers", "-q"])  # type: ignore
    try:
        import PIL  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow", "-q"])  # type: ignore

    if not os.path.exists('AI-og-helse'):
        print("📥 Laster ned kursmateriell...")
        subprocess.check_call(["git", "clone", "https://github.com/arvidl/AI-og-helse.git"])  # type: ignore
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Byttet til mappe: {os.getcwd()}")
else:
    print("💻 Kjører i lokalt miljø")

import numpy as np
import torch
from PIL import Image, ImageOps, ImageFilter

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("✅ Miljø klart")


In [ ]:
# Konfigurasjon
LABELS = ["daisy","dandelion","rose","sunflower","tulip"]
DISTRACTOR_LABELS = ["chrysanthemum","bouquet","wildflower","garden flower"]
PROMPT_TEMPLATES = [
    "a photo of a {}",
    "a close-up photo of a {}",
    "a high quality image of a {}",
    # Norsk/synonymer
    "et foto av en {}",
    "et nærbilde av en {}",
    "et høyoppløselig bilde av en {}"
]

# Finn et utvalg av bilder per klasse
# Gjør path robust ift. arbeidskatalog (notebook-mappe vs. repo-rot)
POSSIBLE_BASE_DIRS = [
    os.path.join("data", "flowers"),
    os.path.join("..", "data", "flowers"),
    os.path.join("..", "..", "data", "flowers"),
]
BASE_DIR = next((p for p in POSSIBLE_BASE_DIRS if os.path.isdir(p)), POSSIBLE_BASE_DIRS[0])
SAMPLES_PER_CLASS = 25  # gjort vanskeligere ved å øke utvalget

def collect_image_paths(base_dir, labels, k=25):
    paths = []
    exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    for label in labels:
        label_dir = os.path.join(base_dir, label)
        found = []
        for ext in exts:
            found.extend(glob.glob(os.path.join(label_dir, ext)))
        found = sorted(found)
        random.shuffle(found)
        paths.extend([(pp, label) for pp in found[:k]])
    return paths

image_label_pairs = collect_image_paths(BASE_DIR, LABELS, SAMPLES_PER_CLASS)
print(f"Bruker bildemappe: {os.path.abspath(BASE_DIR)}")
print(f"Antall bilder valgt: {len(image_label_pairs)}")
if len(image_label_pairs) == 0:
    print("⚠️ Fant ingen bilder. Sjekk at blomsterdatasettet er tilgjengelig under 'data/flowers'.")
    print("💡 Tips: I lokal kjøring starter notatboken ofte i mappen 'uke04-generativ-ai'.")
    print("   Prøv å flytte arbeidskatalogen til repo-roten, eller oppdater stien til '../data/flowers'.")
    print("   Se også uke03-notebooks (02a–02d) for detaljer om dataoppsett.")


In [ ]:
# Last modell og prosessor

# Importer lokalt i denne cellen for å tåle at celler kjøres i feil rekkefølge
import os
import sys
import subprocess
import torch

# Prøv å importere Hugging Face-varianten først
try:
    from transformers import CLIPProcessor, CLIPModel  # type: ignore
except Exception:
    CLIPProcessor = None
    CLIPModel = None

# Velg enhet: CUDA > MPS (Apple) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Skru av HF Transfer (kan gi 500-feil i noen miljøer)
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

BACKEND = None
MODEL_ID = None
model = None
processor = None
openai_clip = None
openai_clip_preprocess = None
last_err = None

# 1) Foretrekk OpenAI CLIP-implementasjonen (unngår HF 500-feil)
try:
    import clip as openai_clip  # type: ignore
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/openai/CLIP.git", "-q"])  # type: ignore
        import clip as openai_clip  # type: ignore
    except Exception as e:
        last_err = e
        openai_clip = None

if openai_clip is not None and model is None:
    try:
        model, openai_clip_preprocess = openai_clip.load("ViT-B/32", device=DEVICE, jit=False)
        BACKEND = "openai"
        MODEL_ID = "ViT-B/32"
        print(f"Modell '{MODEL_ID}' lastet på {DEVICE} (OpenAI CLIP)")
    except Exception as e:
        last_err = e

# 2) Fallback til Hugging Face hvis OpenAI-CLIP ikke er tilgjengelig
if model is None:
    try:
        from transformers import CLIPProcessor, CLIPModel  # type: ignore
        for mid in [
            "openai/clip-vit-base-patch32",
            "openai/clip-vit-base-patch16",
        ]:
            try:
                model = CLIPModel.from_pretrained(mid).to(DEVICE)
                processor = CLIPProcessor.from_pretrained(mid)
                MODEL_ID = mid
                BACKEND = "hf"
                print(f"Modell '{mid}' lastet på {DEVICE} (Hugging Face)")
                break
            except Exception as e:
                last_err = e
                print(f"⚠️ Klarte ikke å laste '{mid}': {e}")
    except Exception as e:
        last_err = e

if model is None:
    raise RuntimeError(
        "Kunne ikke laste CLIP-modell.\n"
        "Prøv igjen senere, sjekk nettverk, eller oppgrader 'transformers' / 'huggingface_hub'.\n"
        "Alternativt last ned modellen manuelt.\n"
        f"Siste feil: {last_err}"
    )


In [ ]:
# Hjelpefunksjoner

def build_prompts(labels, templates):
    prompts = []
    for l in labels:
        for t in templates:
            prompts.append(t.format(l))
    return prompts

ALL_PROMPTS = build_prompts(LABELS + DISTRACTOR_LABELS, PROMPT_TEMPLATES)

@torch.no_grad()
def clip_zero_shot(image_path, prompts, *, center_crop=True, resize_to=224, blur_sigma=0.5, rotate_deg=5):
    image = Image.open(image_path).convert("RGB")
    # Gjør oppgaven vanskeligere: lett preprosess
    if center_crop:
        image = ImageOps.fit(image, (min(image.size), min(image.size)), Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    if resize_to is not None:
        image = image.resize((resize_to, resize_to), Image.Resampling.LANCZOS)
    if blur_sigma and blur_sigma > 0:
        image = image.filter(ImageFilter.GaussianBlur(radius=blur_sigma))
    if rotate_deg and rotate_deg != 0:
        image = image.rotate(rotate_deg, resample=Image.Resampling.BICUBIC)

    if BACKEND == "hf":
        inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True).to(DEVICE)
        outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=-1).squeeze(0).detach().cpu()
        return probs
    elif BACKEND == "openai":
        assert openai_clip_preprocess is not None, "Mangler OpenAI CLIP preprocess"
        image_input = openai_clip_preprocess(image).unsqueeze(0).to(DEVICE)
        text_tokens = openai_clip.tokenize(prompts).to(DEVICE)
        logits_per_image, logits_per_text = model(image_input, text_tokens)
        probs = logits_per_image.softmax(dim=-1).squeeze(0).detach().cpu()
        return probs
    else:
        raise RuntimeError("Ukjent CLIP-backend")

# Kjør prediksjon på alle bilder og beregn enkel accuracy
results = []
for img_path, true_label in image_label_pairs:
    probs = clip_zero_shot(img_path, ALL_PROMPTS,
                           center_crop=True, resize_to=224,
                           blur_sigma=0.8, rotate_deg=7)
    # Aggreger over prompt-varianter per klasse (kun for ekte klasser; ignorer forvirrere)
    num_templates = len(PROMPT_TEMPLATES)
    class_scores = []
    for i, label in enumerate(LABELS):
        # indekser for denne klassens prompts i den utvidede prompt-listen
        start = i * num_templates
        end = (i + 1) * num_templates
        score = probs[start:end].mean().item()
        class_scores.append(score)
    class_scores = torch.tensor(class_scores)
    pred_idx = int(torch.argmax(class_scores))
    pred_label = LABELS[pred_idx]
    results.append((img_path, true_label, pred_label, class_scores.tolist()))

accuracy = sum(1 for _, t, p, _ in results if t == p) / max(1, len(results))
print(f"Enkel accuracy på utvalget: {accuracy:.2%}")


In [ ]:
# Visualisering av flere eksempler
import matplotlib.pyplot as plt
import math

num_show = min(12, len(results))
if num_show == 0:
    print("Ingen resultater å vise.")
else:
    cols = min(4, max(1, num_show))
    rows = math.ceil(num_show / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows))
    if hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for ax, (img_path, true_label, pred_label, class_scores) in zip(axes, results[:num_show]):
        ax.imshow(Image.open(img_path))
        ax.set_title(f"GT: {true_label}\nPred: {pred_label}")
        ax.axis('off')
    for ax in axes[num_show:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Vis topp-5 for de første 5 bildene (eller færre hvis mindre datasett)
if len(results) > 0:
    n_toplist = min(5, len(results))
    topk = min(5, len(LABELS))
    print(f"Top-5 klasser for de første {n_toplist} bildene:")
    for idx, (_, true_label, pred_label, class_scores) in enumerate(results[:n_toplist], start=1):
        scores = torch.tensor(class_scores)
        vals, idxs = torch.topk(scores, topk)
        print(f"[{idx}] GT={true_label}, Pred={pred_label}")
        for v, i in zip(vals.tolist(), idxs.tolist()):
            print(f"  {LABELS[i]}: {v:.3f}")


### Veiledning: Interaktiv zero-shot prompting

- Hva cellen gjør
  - Viser et lite brukergrensesnitt for zero-shot‑klassifikasjon med CLIP.
  - To moduser:
    - Rå promter: du skriver egne tekstlinjer; modellen rangerer disse direkte mot valgt bilde.
    - Klasser + maler: du oppgir klassenavn (kommaseparert) og maler med {label}; promter genereres og aggregeres per klasse.

- Slik bruker du den
  1) Velg et bilde i nedtrekksmenyen Bilde.
  2) Velg én av to moduser:
     - Rå promter: skriv én prompt per linje i feltet Rå promter. Ignorer feltene Klasser og Maler.
     - Klasser + maler: la Rå promter være tomt, fyll inn Klasser (kommaseparert) og Maler (én per linje). Bruk {label} i malene.
  3) Juster Top‑k for hvor mange topplasseringer som vises.
  4) Klikk Kjør.

- Hva som skjer i hver modus
  - Rå promter:
    - Modellen scorer hver individuell tekstlinje direkte mot bildet (ingen aggregering).
    - Top‑k viser hvilke promter som passer best (horisontal stolpediagram).
  - Klasser + maler:
    - For hver klasse dannes promter ved å sette inn {label} i hver mal.
    - Sannsynligheter aggregeres per klasse (gjennomsnitt over maler) før rangering.
    - Viser predikert klasse og top‑k klasser (horisontal stolpediagram).

- Eksempel 1: Rå promter
```
a photo of a sunflower
a macro shot of a daisy
a high quality image of a tulip
```

- Eksempel 2: Klasser + maler
Klasser:
```
daisy,sunflower,tulip,rose,dandelion
```
Maler:
```
a photo of a {label}
a close-up photo of a {label}
a high quality image of a {label}
```

- Tips til gode promter
  - Start enkelt: "a photo of a {label}".
  - Legg til kontekst forsiktig: "a close-up …", "in a garden", "with green leaves".
  - Unngå for spesifikke/tvetydige ord.
  - Bruk engelsk for best kompatibilitet med CLIP.

- Tolkning av output
  - Pred: klassen (eller prompten) med høyest score.
  - Top‑k: bar‑plott med høyeste sannsynligheter; jevne verdier tyder på usikkerhet/forveksling.
  - I Rå promter‑modus er “klassen” i grafen selve tekstlinjen du skrev.

- Feilsøking
  - «Ingen bilder funnet»: legg bilder i data/flowers/<klasse> (i Colab: last opp mappen).
  - Tom/rar output: prøv enklere promter; øk antall maler; sjekk klasser‑streng for ekstra mellomrom.
  - Top‑k > antall promter/klasser: begrenses automatisk.

- Ytelse
  - Kjører på CPU/MPS/CUDA. Flere promter/klasser gir lengre kjøretid.


### Batch‑modus: Prompt hele samlingen av bilder

Bruk denne seksjonen når du ønsker å kjøre de samme promtene mot alle bilder i utvalget (uten å velge ett bilde først). Du kan:
- enten skrive Rå promter (én per linje)
- eller bruke Klasser + Maler (slik som i den interaktive cellen over)

Klikk «Kjør batch» for å få sammendrag: accuracy (dersom klassenavn matcher datasett‑klasser), samt topp‑resultater og et lite sammendrag av fordelingen.


In [ ]:
# Batch‑modus: kjør promter på hele utvalget
from ipywidgets import Button, HTML, VBox

batch_status = HTML("")
batch_btn = Button(description="Kjør batch", button_style="info")

def run_batch(_):
    batch_status.value = ""
    if len(image_label_pairs) == 0:
        batch_status.value = "<b>Ingen bilder funnet.</b> Sørg for at data/flowers er tilgjengelig."
        return

    # Les innstillinger fra eksisterende widgets (gjenbruk av samme felter)
    raw_lines = []
    try:
        raw_lines = [ln.strip() for ln in raw_prompts_ta.value.splitlines() if ln.strip()]
    except Exception:
        pass

    # Velg modus
    use_raw = len(raw_lines) > 0
    if use_raw:
        prompts = raw_lines
    else:
        labels = [s.strip() for s in labels_txt.value.split(',') if s.strip()]
        templates = [ln.strip() for ln in templates_ta.value.splitlines() if ln.strip()]
        if len(labels) == 0 or len(templates) == 0:
            batch_status.value = "<b>Mangler klasser eller maler.</b> Fyll inn begge, eller bruk Rå promter."
            return
        prompts = []
        for l in labels:
            for t in templates:
                prompts.append(t.format(l))
        K = len(templates)

    results_batch = []
    correct = 0
    for img_path, true_label in image_label_pairs:
        probs = clip_zero_shot(img_path, prompts)
        if use_raw:
            # Ingen aggregering – finn topp‑prompt
            pred_idx = int(torch.argmax(probs))
            # Forsøk å mappe prompt til klasse (kun hvis prompt er nøyaktig klassenavn i tekst)
            pred_label = prompts[pred_idx]
        else:
            # Aggreger per klasse
            class_scores = []
            for j, _ in enumerate(labels):
                start = j * K
                end = (j + 1) * K
                class_scores.append(probs[start:end].mean().item())
            class_scores_t = torch.tensor(class_scores)
            pred_idx = int(torch.argmax(class_scores_t))
            pred_label = labels[pred_idx]
        results_batch.append((img_path, true_label, pred_label))
        if pred_label == true_label:
            correct += 1

    msg = []
    if not use_raw:
        acc = correct / max(1, len(image_label_pairs))
        msg.append(f"Accuracy: {acc:.2%} ({correct}/{len(image_label_pairs)})")
    else:
        msg.append("Kjørte batch i rå‑prompt‑modus (ingen aggregert accuracy).")

    # Vis noen få linjer
    preview = "\n".join([f"- {os.path.basename(p)}: GT={t}, Pred={pl}" for p, t, pl in results_batch[:10]])
    batch_status.value = "<pre>" + "\n".join(msg) + "\n\n" + preview + ("\n…" if len(results_batch) > 10 else "") + "</pre>"

batch_btn.on_click(run_batch)
display(VBox([batch_btn, batch_status]))


In [ ]:
# Interaktiv prompting med ipywidgets
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Text, Textarea, Dropdown, Button, IntSlider, HTML
from IPython.display import display

# Bygg bildeforslag fra eksisterte paths
image_options = [(os.path.basename(p), p) for p, _ in image_label_pairs] if len(image_label_pairs) > 0 else []

image_dd = Dropdown(options=image_options, description="Bilde:", layout={"width": "400px"})
labels_txt = Text(value=",".join(LABELS), description="Klasser:", layout={"width": "400px"})
templates_ta = Textarea(value="\n".join(PROMPT_TEMPLATES), description="Maler:", layout={"width": "400px", "height": "100px"})
raw_prompts_ta = Textarea(value="", description="Rå promter:", placeholder="En prompt per linje (valgfritt)", layout={"width": "400px", "height": "100px"})

topk_slider = IntSlider(value=5, min=1, max=10, step=1, description="Top-k:")
run_btn = Button(description="Kjør", button_style="primary")
status_html = HTML(value="")

out = HTML("")


def run_inference(_):
    status_html.value = ""
    if len(image_label_pairs) == 0 or image_dd.value is None:
        status_html.value = "<b>Ingen bilder funnet.</b> Sørg for at data/flowers er tilgjengelig."
        return
    img_path = image_dd.value

    # Velg modus: rå promter eller labels+maler
    raw_lines = [ln.strip() for ln in raw_prompts_ta.value.splitlines() if ln.strip()]
    if len(raw_lines) > 0:
        prompts = raw_lines
        # Kjør direkte uten aggregering
        probs = clip_zero_shot(img_path, prompts)
        k = min(topk_slider.value, len(prompts))
        vals, idxs = torch.topk(probs, k)
        # Render
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].imshow(Image.open(img_path))
        ax[0].axis('off')
        ax[0].set_title("Valgt bilde")
        ylabels = [prompts[i] for i in idxs.tolist()][::-1]
        yvals = vals.tolist()[::-1]
        ax[1].barh(range(len(yvals)), yvals)
        ax[1].set_yticks(range(len(yvals)))
        ax[1].set_yticklabels(ylabels)
        ax[1].set_title("Top-k promter")
        ax[1].set_xlim(0, 1)
        plt.tight_layout()
        plt.show()
        status_html.value = f"Kjørte {len(prompts)} rå promter."
        return

    # Labels + maler med aggregering per klasse
    labels = [s.strip() for s in labels_txt.value.split(',') if s.strip()]
    templates = [ln.strip() for ln in templates_ta.value.splitlines() if ln.strip()]
    if len(labels) == 0 or len(templates) == 0:
        status_html.value = "<b>Mangler klasser eller maler.</b> Fyll inn begge, eller bruk Rå promter."
        return

    prompts = []
    for l in labels:
        for t in templates:
            prompts.append(t.format(l))

    probs = clip_zero_shot(img_path, prompts)

    # Aggreger per klasse (gjennomsnitt over maler)
    K = len(templates)
    class_scores = []
    for j, _ in enumerate(labels):
        start = j * K
        end = (j + 1) * K
        class_scores.append(probs[start:end].mean().item())
    class_scores_t = torch.tensor(class_scores)
    pred_idx = int(torch.argmax(class_scores_t))
    pred_label = labels[pred_idx]

    # Visualisering
    k = min(topk_slider.value, len(labels))
    vals, idxs = torch.topk(class_scores_t, k)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(Image.open(img_path))
    ax[0].axis('off')
    ax[0].set_title(f"Pred: {pred_label}")

    ylabels = [labels[i] for i in idxs.tolist()][::-1]
    yvals = vals.tolist()[::-1]
    ax[1].barh(range(len(yvals)), yvals)
    ax[1].set_yticks(range(len(yvals)))
    ax[1].set_yticklabels(ylabels)
    ax[1].set_title("Top-k klasser")
    ax[1].set_xlim(0, 1)
    plt.tight_layout()
    plt.show()

    status_html.value = f"Aggregerte {len(templates)} maler per klasse over {len(labels)} klasser. Pred: <b>{pred_label}</b>."


run_btn.on_click(run_inference)

ui = VBox([
    HTML("<h3>Interaktiv zero-shot prompting</h3>"),
    HBox([image_dd, topk_slider]),
    HTML("<b>Alternativ 1:</b> Bruk rå promter (én per linje) – ignorerer klasser/maler."),
    raw_prompts_ta,
    HTML("<b>Alternativ 2:</b> Bruk klasser og maler. {label} i malene erstattes med klassens navn."),
    labels_txt,
    templates_ta,
    run_btn,
    status_html
])

display(ui)


In [ ]:
print(len(image_label_pairs), BACKEND, MODEL_ID)

### Refleksjon
- Hvorfor kan noen prompt-varianter fungere bedre enn andre?
- Når svikter zero-shot klassifikasjon på dette datasettet?
- Hvordan påvirker antall prompt-varianter resultatet?


### Hva notebooken gjør (kort)
- Kjører zero-shot bildeklassifikasjon på blomsterbilder med CLIP, uten fintrening.
- Sammenligner hvert bilde mot en liten liste tekstprompter per klasse og velger klassen med høyest sannsynlighet.
- Viser enkel nøyaktighet på et lite utvalg og visualiserer noen eksempler.

### Rasjonale
- CLIP lærer felles representasjoner for tekst og bilde. Zero-shot betyr at vi kan klassifisere nye klasser ved å formulere dem som tekstprompter.
- Pedagogisk verdi: viser multimodal forståelse, prompt-sensitivitet og begrensninger ved å bruke tekst som “etiketter”.

### Data og oppsett
- **Datasett**: `data/flowers/{daisy,dandelion,rose,sunflower,tulip}`.
- Velger et lite antall bilder per klasse for rask kjøring.
- **Robuste stier**: forsøker `data/flowers`, `../data/flowers`, `../../data/flowers`; støtter `.jpg/.jpeg/.png` (og store bokstaver).
- Skriver ut brukt mappe og antall bilder; gir veiledning hvis 0 bilder.

### Modell og backend-strategi
- **Primært**: OpenAI-CLIP (Python-implementasjon) for å unngå midlertidige 500-feil fra Hugging Face Hub.
- **Fallback**: Hugging Face `openai/clip-vit-base-patch32` (evt. patch16) via `transformers`.
- **Enhet**: prioriterer `cuda` > `mps` (Apple Silicon) > `cpu`.
- **Miljøflagg**: deaktiverer HF Transfer-lag (`HF_HUB_ENABLE_HF_TRANSFER=0`) som i noen miljøer kan gi 500-feil ved nedlasting.

### Prompt-design og zero-shot-metode
- Klassenavn: `daisy`, `dandelion`, `rose`, `sunflower`, `tulip`.
- Prompt-ensemble:
  - “a photo of a {}”
  - “a close-up photo of a {}”
  - “a high quality image of a {}”
- For hver klasse genereres flere promter. Modellens sannsynligheter aggregeres per klasse (snitt) før prediksjon velges.

#### Eksplicitte prompt-strenger (eksempler)
For et bilde evalueres følgende 15 tekstprompter (3 per klasse):

```text
# daisy
"a photo of a daisy"
"a close-up photo of a daisy"
"a high quality image of a daisy"

# dandelion
"a photo of a dandelion"
"a close-up photo of a dandelion"
"a high quality image of a dandelion"

# rose
"a photo of a rose"
"a close-up photo of a rose"
"a high quality image of a rose"

# sunflower
"a photo of a sunflower"
"a close-up photo of a sunflower"
"a high quality image of a sunflower"

# tulip
"a photo of a tulip"
"a close-up photo of a tulip"
"a high quality image of a tulip"
```

### Inferenz- og evalueringsløp
- For hvert bilde:
  - Kjør CLIP med alle tekstprompter (rekkefølgen over).
  - Modellens output er logitter/sannsynligheter for hver prompt i samme rekkefølge.
  - Aggreger per klasse: ta gjennomsnitt av sannsynligheter for de tre promptene som hører til samme klasse.
  - Velg klassen med høyest aggregerte sannsynlighet (top-1).
- **Evalueringsmål**: enkel accuracy på det lille utvalget.
- **Visualisering**: rutenett med bilde, GT-klasse og predikert klasse; top-5-liste for første eksempel.

#### Svarformat og tolkning (eksplisitt)
- Rå-output: en vektor med sannsynligheter `p(prompt_i | bilde)` for alle 15 promter.
- Etter aggregering får vi `p(klasse_j | bilde)` ved å snitte de tre promtene per klasse.
- Rapportert top-5 (for første bilde) er sorterte `(klasse, sannsynlighet)`-par, for eksempel:

```text
Top-5 klasser:
  sunflower: 0.62
  daisy: 0.18
  tulip: 0.11
  rose: 0.06
  dandelion: 0.03
```

### Robusthetstiltak i koden
- **Filsystem**: tåler forskjellig arbeidskatalog; støtter flere filendelser.
- **Modelllasting**: forsøker OpenAI-CLIP først; om nødvendig installerer og laster. Faller tilbake til HF-modeller i prioritert rekkefølge.
- **Reproduserbarhet**: setter seeds (`random`, `numpy`, `torch`). Importerer i nøkkelceller for å tåle feil kjøre-rekkefølge.

### Funn og hvordan lese output
- “Bruker bildemappe: …” og “Antall bilder valgt: …” bekrefter data.
- “Modell '…' lastet på … (OpenAI CLIP/Hugging Face)” viser hvilken backend som brukes.
- “Enkel accuracy på utvalget: X%” gir grovt estimat (ikke statistisk robust).
- Top-5-listen for første bilde viser sannsynlighetsfordelingen (usikkerhet).

### Tolkning
- God presisjon når motivene er tydelige; prompt-formulering påvirker resultat.
- Prompt-ensemble gir mer robusthet enn én prompt.
- Feil typisk for visuelt like klasser, harde utsnitt, eller “støyete” bilder.

### Begrensninger
- Ingen fintrening på blomster; ren zero-shot.
- Lite evalueringsutvalg ⇒ resultater er demonstrative.
- Sensitivt for promptvalg; enkle engelske beskrivelser brukes.

### Forslag til videre arbeid
- Flere/bredere promter per klasse (inkl. norsk/engelsk synonymer).
- Vektet aggregering eller temperatur-skalering av sannsynligheter.
- Øk utvalget og gjør stratifisert sampling; legg til top-k accuracy og forvekslingsmatrise.
- Sammenlign CLIP-varianter (ViT-B/16, RN50) når HF Hub er stabil.

### Praktiske tips
- Hvis HF laster tregt/feiler: kjør via OpenAI-CLIP (default her).
- Lokalt: sørg for at `data/flowers` ligger i repo-rot, eller juster stien.
- Colab: repo klones og `os.chdir('AI-og-helse')`; last opp `data/flowers` ved behov.


## Tre grunnleggende referanser om CLIP:

### 1. **OpenAI's offisielle CLIP-introduksjon**
[OpenAI - CLIP: Connecting text and images](https://openai.com/index/clip/)

Dette er den offisielle introduksjonen fra OpenAI som forklarer at CLIP er et nevralt nettverk som effektivt lærer visuelle konsepter fra naturlig språk. Modellen kan anvendes på enhver visuell klassifiseringsoppgave ved kun å gi navnene på de visuelle kategoriene som skal gjenkjennes, likt "zero-shot"-evnene til GPT-2 og GPT-3.

### 2. **Hugging Face Community kurs**
[Contrastive Language-Image Pre-training (CLIP)](https://huggingface.co/learn/computer-vision-course/en/unit4/multimodal-models/clip-and-relatives/clip)

Dette er en pedagogisk ressurs som forklarer at CLIP er et nevralt nettverk som er dyktig til å forstå visuelle konsepter gjennom naturlig språkveiledning. Den trener samtidig en tekstkoder og en bildekoder, med fokus på en forhåndstreningsoppgave som handler om å matche bildetekster med tilsvarende bilder.

### 3. **GitHub - OpenAI CLIP repository**
[GitHub - openai/CLIP](https://github.com/openai/CLIP)

Den offisielle GitHub-repositorien inneholder både kode og dokumentasjon som forklarer at CLIP er et nevralt nettverk trent på ulike bilde-tekst-par. Den kan instrueres på naturlig språk til å forutsi den mest relevante tekstbeskrivelsen for et gitt bilde, uten direkte optimalisering for oppgaven, på samme måte som zero-shot-evnene til GPT-modellene.

Disse tre kildene gir en god introduksjon til CLIP på ulike nivåer - fra den offisielle produktsiden, via et pedagogisk kurs, til den tekniske implementasjonen på GitHub.